# Parte 2 · Clasificación de titulares

Esta parte toma los 384 titulares de `datos/noticias.csv` (Parte 1), los etiqueta con un LLM y entrena con esas etiquetas un Transformer ligero en español. El modelo resultante se publicará en Hugging Face Hub para que el dashboard de la Parte 3 lo cargue desde ahí.

La columna `etiqueta` del CSV de la Parte 1 dice en qué sección publicó el medio la nota, no de qué trata el titular: es una **etiqueta débil**. Por eso el entrenamiento usa las etiquetas del LLM, y la etiqueta por sección solo se usa para compararlas.

Pasos:

1. Cargar los datos.
2. Etiquetar las 384 noticias con un LLM, en lotes.
3. Comparar la etiqueta por sección con la del LLM.
4. Auditoría humana de 20 casos.
5. Split estratificado 70/15/15.
6. Baseline zero-shot sobre 40 titulares del conjunto de test.
7. Fine-tune de un Transformer ligero en español.
8. Métricas en test y comparación entre zero-shot y fine-tune.
9. Guardar el modelo y dejar listo el código para subirlo a Hugging Face Hub.

El baseline zero-shot va después del split, y no al principio, para que se evalúe sobre los mismos titulares y contra las mismas etiquetas que el fine-tune. Si se calculara antes, sobre 40 titulares cualquiera, las dos cifras no serían comparables y algunos de esos titulares podrían terminar en el entrenamiento.

## 0. Librerías y constantes

`SEMILLA` fija todo lo que depende del azar en este notebook: la muestra de la auditoría, el split y el entrenamiento.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.metrics import cohen_kappa_score

SEMILLA = 42
CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
DATOS = Path("datos")

pd.set_option("display.max_colwidth", None)

## 1. Carga de los datos

Las 384 noticias de la Parte 1, con su etiqueta por sección.

In [2]:
noticias = pd.read_csv(DATOS / "noticias.csv", encoding="utf-8-sig")
print(f"{len(noticias)} noticias")
noticias["etiqueta"].value_counts().reindex(CATEGORIAS).rename("titulares por sección").to_frame()

384 noticias


,titulares por sección
etiqueta,
politica,92
economia,99
deportes,98
ciencia_tecnologia,95


## 2. Etiquetado con un LLM

El LLM que etiquetó es Claude (Claude Opus 5.5), en la misma sesión de trabajo en que se armó este notebook, el 18 de octubre de 2026. El notebook no puede llamar al modelo, así que el etiquetado se hizo fuera de él y el resultado quedó en `datos/etiquetas_llm.csv`. La celda de abajo lo carga y comprueba que esté completo.

Procedimiento:

- Los 384 titulares se mezclaron con la semilla 42 y se dividieron en 14 lotes: 6 de 28 y 8 de 27. Mezclarlos evita que un lote quede lleno de titulares de una misma sección.
- En cada llamada el modelo recibió un lote completo, con un identificador y el titular de cada noticia. No vio la bajada, la sección ni la URL (que lleva la sección en la ruta), para que la etiqueta dependa solo del texto que después verá el clasificador.
- Las instrucciones fueron las mismas para todos los lotes:

> Clasifica cada titular de una noticia peruana en exactamente una de estas categorías: `politica`, `economia`, `deportes`, `ciencia_tecnologia`. Usa solo el texto del titular.
>
> - `politica`: Poder Ejecutivo, Congreso, partidos, elecciones, autoridades, casos judiciales o fiscales que involucran a políticos o al Estado, y política internacional.
> - `economia`: mercados, tipo de cambio, precios, empresas, inversión, empleo, comercio, minería, energía y pesca como actividades económicas, impuestos, pensiones y consumo.
> - `deportes`: competencias, clubes, selecciones, deportistas, resultados, fichajes y transmisiones.
> - `ciencia_tecnologia`: investigación y descubrimientos, salud vista desde la ciencia, medio ambiente y clima, espacio, tecnología, internet, redes sociales, inteligencia artificial, dispositivos, videojuegos y ciberseguridad.
>
> Si un titular encaja en dos categorías, elige la del tema central: por ejemplo, una ley de pensiones va a `politica` si el titular trata del trámite o los votos, y a `economia` si trata del efecto en el bolsillo. No hay categoría "otra": si ninguna encaja del todo, elige la más cercana.

Una limitación del procedimiento: el identificador de cada titular era su posición en `noticias.csv`, y ese archivo está ordenado por sección, así que la posición se relaciona con la sección. Las etiquetas se decidieron por el texto, pero un etiquetado estrictamente ciego usaría identificadores al azar.

In [3]:
etiquetas_llm = pd.read_csv(DATOS / "etiquetas_llm.csv", encoding="utf-8-sig")
por_lote = etiquetas_llm.groupby("lote").size()

assert len(etiquetas_llm) == len(noticias) and etiquetas_llm["url"].is_unique
assert set(etiquetas_llm["url"]) == set(noticias["url"])
assert set(etiquetas_llm["etiqueta_llm"]) <= set(CATEGORIAS)
assert por_lote.between(25, 30).all()

df = (noticias.rename(columns={"etiqueta": "etiqueta_seccion"})
              .merge(etiquetas_llm[["url", "etiqueta_llm", "lote"]], on="url", validate="one_to_one"))
print(f"{len(por_lote)} lotes de entre {por_lote.min()} y {por_lote.max()} titulares")
df["etiqueta_llm"].value_counts().reindex(CATEGORIAS).rename("titulares según el LLM").to_frame()

14 lotes de entre 27 y 28 titulares


,titulares según el LLM
etiqueta_llm,
politica,97
economia,96
deportes,98
ciencia_tecnologia,93


## 3. Etiqueta por sección frente a etiqueta del LLM

La tabla cruza las dos etiquetas: las filas son la sección y las columnas, el LLM, así que la diagonal reúne las coincidencias. Además del porcentaje de acuerdo se calcula el kappa de Cohen, que descuenta el acuerdo que se daría por azar (1 es acuerdo perfecto; 0, el que daría el azar). Debajo van todos los titulares en que no coinciden.

In [4]:
coinciden = df["etiqueta_seccion"] == df["etiqueta_llm"]
print(f"Coinciden: {coinciden.sum()} de {len(df)} ({coinciden.mean():.1%})")
print(f"No coinciden: {(~coinciden).sum()}")
print(f"Kappa de Cohen: {cohen_kappa_score(df['etiqueta_seccion'], df['etiqueta_llm']):.3f}")

display(pd.crosstab(df["etiqueta_seccion"], df["etiqueta_llm"], rownames=["sección"], colnames=["LLM"],
                    margins=True, margins_name="total")
          .reindex(index=CATEGORIAS + ["total"], columns=CATEGORIAS + ["total"]))
(df.loc[~coinciden, ["titular", "seccion_original", "etiqueta_seccion", "etiqueta_llm"]]
   .sort_values(["etiqueta_seccion", "etiqueta_llm"], ignore_index=True))

Coinciden: 370 de 384 (96.4%)
No coinciden: 14


Kappa de Cohen: 0.951


LLM,politica,economia,deportes,ciencia_tecnologia,total
sección,,,,,
politica,91,1,0,0,92
economia,1,93,0,5,99
deportes,0,0,98,0,98
ciencia_tecnologia,5,2,0,88,95
total,97,96,98,93,384


,titular,seccion_original,etiqueta_seccion,etiqueta_llm
0,"Netflix despedirá a unos 1.000 empleados, el 6 % de su plantilla, según medios",tecnologia/actualidad,ciencia_tecnologia,economia
1,Menos comida y más cara: El Niño amenaza los alimentos y la pesca en Latinoamérica,tecnologia/ecologia,ciencia_tecnologia,economia
2,El Niño en Perú: no está operativo ninguno de los 18 sistemas de alertas tempranas previstos desde 2017,tecnologia/ecologia,ciencia_tecnologia,politica
3,"Demetrio Pacheco, defensor ambiental en Perú: “El Estado podría haber evitado el asesinato de mi hijo” | ENTREVISTA",tecnologia/ecologia,ciencia_tecnologia,politica
4,Trump quiere llamar “superinteligencia” a la IA y amenaza con tratar como “enemigo” a quien no lo haga,tecnologia/actualidad,ciencia_tecnologia,politica
5,Trump llama a Musk un “tesoro nacional” tras condecorarlo con galardón científico de Estados Unidos,tecnologia/actualidad,ciencia_tecnologia,politica
6,"Estados Unidos suspenderá a Microsoft y a Adobe de programa de visados, anuncia JD Vance",tecnologia/actualidad,ciencia_tecnologia,politica
7,IA y Sandbox regulatorio,economia/opinion,economia,ciencia_tecnologia
8,Indecopi se pronuncia sobre presunta filtración de datos en Interseguro,economia,economia,ciencia_tecnologia
9,Creación de agentes de IA se multiplicó por 15 en el último año: ¿cómo impacta en la productividad empresarial?,economia/dia-1,economia,ciencia_tecnologia


**Qué muestran las diferencias.** Las 14 discrepancias son casos de frontera, en que la sección y el tema del titular se separan:

- **Deportes** coincide en todos los casos.
- **Economía → ciencia y tecnología (5)**: la filtración de datos de Interseguro (tres notas que el medio publicó en economía porque la afectada es una aseguradora, y que el LLM clasifica como ciberseguridad) y dos piezas sobre inteligencia artificial.
- **Ciencia y tecnología → política (5) o economía (2)**: notas publicadas en tecnología o ecología cuyo tema central es un gobierno (dos sobre Trump, las visas de Microsoft y Adobe, los sistemas de alerta ante El Niño, el asesinato de un defensor ambiental) o un efecto económico (despidos en Netflix, el precio de los alimentos).
- **Entre política y economía (2)**: la mesa técnica entre la Unión Europea y el Minem para destrabar permisos (política → economía) y la columna «Invocación por el Perú» (economía → política), cuyo título apenas da pistas del tema.

El clasificador se entrenará con el criterio del LLM, que se fija en el tema del titular y no en la sección.